In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import torch

# Force CUDA re-initialisation
if torch.cuda.is_available():
    torch.cuda.init()

print(torch.cuda.device_count())  # should now be > 0

In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM
from typing import Optional, Iterable, Dict, List
import os

import pandas as pd
import numpy as np
import warnings
import re
import matplotlib.pyplot as plt
import seaborn as sns
from cmcrameri import cm
import massbalancemachine as mbm
import logging
import torch.nn as nn
from skorch.helper import SliceDataset
from datetime import datetime
from skorch.callbacks import EarlyStopping, LRScheduler, Checkpoint
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset
import pickle 
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch 
from matplotlib.lines import Line2D

from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *
from regions.TF_Europe.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(torch.cuda.is_available())   # True
print(torch.cuda.device_count())   # 0  ← wrong$

if torch.cuda.device_count()==0:
    print("No GPU detected. Using CPU.")
    device = torch.device("cpu")

In [ ]:
BASE_DIR = Path(cfg.dataPath) / path_cache / f"CROSS_REGION_CA_ANALYSIS"
# make sure BASE_DIR exists
os.makedirs(BASE_DIR, exist_ok=True)

print(f"Base directory for data: {BASE_DIR}")

In [ ]:
# MONTHLY_COLS = [
#     't2m',
#     'tp',
#     'slhf',
#     'sshf',
#     'ssrd',
#     'fal',
#     'str',
#     'ELEVATION_DIFFERENCE',
# ]
# STATIC_COLS = ['aspect', 'slope', 'svf']

MONTHLY_COLS = [
    't2m',
    'tp',
    'ssrd',
    'str',
    'ELEVATION_DIFFERENCE',
]
STATIC_COLS = ['aspect', 'slope']

feature_columns = MONTHLY_COLS + STATIC_COLS

# Transform data to monthly format (run or load data):
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_EU_US_CANADA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_EU_US_CANADA.nc"),
}
# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

# Transform data to monthly format (run or load data):
paths_HMA = {
    "era5_climate_data":
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_HIGH_MOUNTAIN_ASIA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_HIGH_MOUNTAIN_ASIA.nc"),
}
# Check that all these files exists
for key, path in paths_HMA.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

# vois_climate = [
#     "t2m",
#     "tp",
#     "slhf",
#     "sshf",
#     "ssrd",
#     "fal",
#     "str",
# ]

# vois_topographical = ["aspect", "slope", "svf"]

# Cross-regional modelling:

### Read stakes datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (SJM+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
dfs["11"]

In [ ]:
rgi_regions = dfs.keys()
num_measurements = {}
for rgi_region in rgi_regions:
    src_code_regions = dfs[rgi_region].SOURCE_CODE.unique()
    for src_code in src_code_regions:
        num_measurements[src_code] = len(
            dfs[rgi_region][dfs[rgi_region].SOURCE_CODE == src_code])
num_measurements

### Monthly datasets:

In [ ]:
SOURCE_REGIONS = ['CH']
experiment_region_groups = {
    # "CEU": ["FR", "IT_AT", "CH"],
    "HMA": ["CENTRALASIA", "SOUTHASIAWEST", "SOUTHASIAEAST"]
}
paths_multi = {
    "EU_US_CANADA": {
        "era5_climate_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_monthly_averaged_data_EU_US_CANADA.nc"),
        "geopotential_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_geopotential_pressure_EU_US_CANADA.nc"),
    },
    "HMA": {
        "era5_climate_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_monthly_averaged_data_HIGH_MOUNTAIN_ASIA.nc"),
        "geopotential_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_geopotential_pressure_HIGH_MOUNTAIN_ASIA.nc"),
    },
}

In [ ]:
XREG_PAIRS = [
    # (source, target)
    ("CH", ["FR", "CENTRALASIA", "ISL"]),
]

# Step 1: collect all unique codes across all pairs
all_codes = sorted(
    {code.upper()
     for src, tgts in XREG_PAIRS
     for code in [src] + tgts})

run_flag_by_code = {
    'ALA': False,
    # 'CENTRALASIA': False,
    'CH': False,
    'ISL': False,
    'NOR': False,
    'SJM': False,
    'FR': False,
}
# Step 2: compute monthly data once per code
monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=MONTHLY_COLS,
    vois_topographical=STATIC_COLS,
    region_codes=all_codes,
    run_flag_by_code=run_flag_by_code,
)

XREG_PAIRS = [
    # (source, target)
    ("CH", [
        "CENTRALASIA",
        "ISL",
        "FR",
    ]),
]

# Step 3: assemble train/test pairs from cache — no ERA5 reprocessing
res_xreg_by_source, split_info_by_source = prepare_xreg_pairs_from_cache(
    cfg=cfg,
    monthly_cache=monthly_cache,
    xreg_pairs=XREG_PAIRS,
)

### Domains shifts & feature overlap:

#### Domain shift:

In [ ]:
EXCLUDE_TARGETS = {}
res_all_xreg_by_source = {}

SOURCE_MEMBERS = {
    "CH": ["CH"],
}

# The 4 individual targets we always want (each source excludes itself)
ALL_TARGETS = ["ISL", "FR", "CENTRALASIA"]

for src_region in SOURCE_REGIONS:
    src_members = SOURCE_MEMBERS[src_region]

    # Exclude the source region itself from targets
    target_codes = [t for t in ALL_TARGETS if t not in src_members]

    res_all_xreg = build_xreg_res_all(
        res_xreg=res_xreg_by_source[src_region],
        target_source_codes=target_codes,
        source_col="SOURCE_CODE",
        key_prefix=f"XREG_{src_region}_TO",
        ch_code=src_region,
        region_groups={},  # no group regions anymore
    )

    # Filter out excluded targets
    if EXCLUDE_TARGETS:
        res_all_xreg = {
            k: v
            for k, v in res_all_xreg.items()
            if not any(tgt in k for tgt in EXCLUDE_TARGETS)
        }

    res_all_xreg_by_source[src_region] = res_all_xreg
    print(f"{src_region} -> keys:", list(res_all_xreg.keys()))

RECOMPUTE_SHIFTS = False

SHIFTS_CACHE = BASE_DIR / "domain_shifts_cache_basic.pkl"

if RECOMPUTE_SHIFTS and SHIFTS_CACHE.exists():
    SHIFTS_CACHE.unlink()
    print(f"Deleted existing cache: {SHIFTS_CACHE}")

if SHIFTS_CACHE.exists():
    with open(SHIFTS_CACHE, "rb") as f:
        cache = pickle.load(f)
    scaler_m = cache["scaler_m"]
    scaler_s = cache["scaler_s"]
    blur_m = cache["blur_m"]
    blur_s = cache["blur_s"]
    blur_joint = cache["blur_joint"]
    bandwidths_m = cache["bandwidths_m"]
    bandwidths_s = cache["bandwidths_s"]
    all_shifts_by_source = cache["all_shifts_by_source"]
    print(
        f"Blurs from cache: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, blur_joint={blur_joint:.4f}"
    )
    print(f"Loaded shifts from cache: {SHIFTS_CACHE}")

else:
    scaler_m, scaler_s, scaler_joint = build_global_scalers_multi_source_simple(
        res_xreg_by_source,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
    )

    blur_m, blur_s, blur_joint = estimate_global_bandwidths_simple(
        res_xreg_by_source,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        scaler_m=scaler_m,
        scaler_s=scaler_s,
        seed=cfg.seed,
    )
    bandwidths_m = [blur_m * 0.5, blur_m, blur_m * 2.0]
    bandwidths_s = [blur_s * 0.5, blur_s, blur_s * 2.0]
    print(
        f"Estimated blurs: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, blur_joint={blur_joint:.4f}"
    )

    all_shifts_by_source = {}
    for src_region in SOURCE_REGIONS:
        all_shifts = {}
        for key in tqdm(res_all_xreg_by_source[src_region], desc=src_region):
            shift = compute_domain_shift(
                df_src=res_all_xreg_by_source[src_region][key]["df_train"],
                df_tgt=res_all_xreg_by_source[src_region][key]["df_test"],
                monthly_cols=MONTHLY_COLS,
                static_cols=STATIC_COLS,
                scaler_m=scaler_m,
                scaler_s=scaler_s,
                scaler_joint=scaler_joint,
                blur_m=blur_m,
                blur_s=blur_s,
                blur_joint=blur_joint,
                bandwidths_m=bandwidths_m,
                bandwidths_s=bandwidths_s,
                seed=cfg.seed,
            )
            all_shifts[key] = shift
        all_shifts_by_source[src_region] = all_shifts

    with open(SHIFTS_CACHE, "wb") as f:
        pickle.dump(
            {
                "scaler_m": scaler_m,
                "scaler_s": scaler_s,
                "blur_m": blur_m,
                "blur_s": blur_s,
                "blur_joint": blur_joint,
                "bandwidths_m": bandwidths_m,
                "bandwidths_s": bandwidths_s,
                "all_shifts_by_source": all_shifts_by_source,
            }, f)
    print(f"Saved shifts to cache: {SHIFTS_CACHE}")

# Plot summary across regions, one figure per source
for src_region, all_shifts in all_shifts_by_source.items():
    print(f"\nDomain shift summary for source: {src_region}")
    fig = plot_domain_shift_across_regions(all_shifts, src_region=src_region)
    plt.show()

In [ ]:
for key in res_all_xreg:
    Num_measurements = res_all_xreg[key]['df_test']['ID'].nunique()
    print(key, 'Target region num_measurements:', Num_measurements)

In [ ]:
from matplotlib.ticker import FuncFormatter
import scipy.stats as scipy_stats


def format_axis_ticks(ax, label_size=8):
    """Format tick labels to avoid huge 1e6/1e7 offset labels."""
    # check if scientific notation offset is being used
    ax.xaxis.get_major_formatter().set_useOffset(False)
    try:
        # scale large numbers to readable units
        xmax = abs(ax.get_xlim()[1])
        if xmax > 1e6:
            scale = 1e6
            ax.xaxis.set_major_formatter(
                FuncFormatter(lambda x, _: f"{x/scale:.1f}"))
            ax.set_xlabel(f"(×10⁶)", label_size=label_size, labelpad=1)
        elif xmax > 1e4:
            scale = 1e3
            ax.xaxis.set_major_formatter(
                FuncFormatter(lambda x, _: f"{x/scale:.0f}"))
            ax.set_xlabel(f"(×10³)", label_size=label_size, labelpad=1)
    except Exception:
        pass


def plot_kde_pair(glaciers_to_plot,
                  selected_cols,
                  save_prefix,
                  ncols=3,
                  nrows=None,
                  figsize=None):
    """KDE panels for a pair of glaciers."""
    if nrows == None:
        nrows = int(np.ceil(len(selected_cols) / ncols))
    if figsize == None:
        w, h = nature_figsize(cols=1, height_mm=160)
        figsize = (w * 2, h)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False)

    legend_handles = []

    for idx, col in enumerate(selected_cols):
        ax = axes[idx // ncols][idx % ncols]

        all_vals = pd.concat([
            cfg_gl["df"][col].dropna() for cfg_gl in glaciers_to_plot.values()
        ])
        x_grid = np.linspace(float(all_vals.min()), float(all_vals.max()), 500)

        for label, cfg_gl in glaciers_to_plot.items():
            vals = cfg_gl["df"][col].dropna().values
            if len(vals) < 10:
                continue
            kde = scipy_stats.gaussian_kde(vals, bw_method=0.3)
            y = kde(x_grid)
            y = y / y.max()
            line, = ax.plot(x_grid,
                            y,
                            color=cfg_gl["color"],
                            linewidth=0.8,
                            label=label)
            ax.fill_between(x_grid, y, alpha=0.08, color=cfg_gl["color"])

            # collect handles only from first panel to avoid duplicates
            if idx == 0:
                legend_handles.append(line)

        ax.set_title(col, fontsize=8)
        ax.set_ylim(0, 1.15)
        ax.set_xlabel("")
        ax.tick_params(labelsize=8, width=0.4, length=2, direction="in")
        ax.spines[["top", "right", "left", "bottom"]].set_visible(True)
        for spine in ax.spines.values():
            spine.set_linewidth(0.4)
        ax.grid(axis="x", color="#e0e0e0", linewidth=0.3)
        ax.set_axisbelow(True)
        format_axis_ticks(ax, label_size=6)

    # ── place legend in first empty axis if one exists, else above figure ──
    empty_axes = [
        axes[idx // ncols][idx % ncols]
        for idx in range(len(selected_cols), nrows * ncols)
    ]

    if empty_axes:
        leg_ax = empty_axes[0]
        leg_ax.axis("off")
        leg_ax.legend(
            handles=legend_handles,
            loc="center",
            fontsize=8,
            frameon=False,
        )
        # turn off remaining empty axes
        for ax in empty_axes[1:]:
            ax.axis("off")
    else:
        # no empty panels — place legend above the figure
        fig.legend(
            handles=legend_handles,
            loc="upper center",
            bbox_to_anchor=(0.5, 1.02),
            ncol=len(glaciers_to_plot),
            fontsize=8,
            frameon=False,
        )

    plt.tight_layout(h_pad=3.0)
    plt.savefig(f"figures/paperTF/{save_prefix}_kde.pdf", bbox_inches="tight")
    plt.savefig(f"figures/paperTF/{save_prefix}_kde.png",
                dpi=300,
                bbox_inches="tight")
    plt.show()

In [ ]:
selected_cols = MONTHLY_COLS + [
    c for c in STATIC_COLS if c != "ELEVATION_DIFFERENCE"
]
selected_cols = selected_cols + ["POINT_BALANCE"]

df_CH = res_all_xreg_by_source['CH']['XREG_CH_TO_ISL']["df_train"]
df_ISL = res_all_xreg_by_source['CH']['XREG_CH_TO_ISL']["df_test"]
glaciers_to_plot = {
    f"CH ": {
        "df": df_CH,
        "color": "#2166ac",  # blue
    },
    f"ISL": {
        "df": df_ISL,
        "color": "#d6604d",  # red
    },
    "CENTRALASIA": {
        "df":
        res_all_xreg_by_source['CH']['XREG_CH_TO_CENTRALASIA']["df_test"],
        "color": "#1a9641",  # green
    },
    "FR": {
        "df": res_all_xreg_by_source['CH']['XREG_CH_TO_FR']["df_test"],
        "color": "#762a83",  # purple
    },
}

plot_kde_pair(
    glaciers_to_plot=glaciers_to_plot,
    selected_cols=selected_cols,
    save_prefix="isl_pool_vs_test",
)

#### Seasonal:

In [ ]:
# Use the actual padded month tokens from the dataset
# Sequence: aug_, sep_, oct, nov, dec, jan, feb, mar, apr, may, jun, jul, aug, sep, oct_

# Get the actual month list from the CH data
from massbalancemachine.data_processing.utils import _rebuild_month_index

month_list_ch, _ = _rebuild_month_index(
    months_head_pad=res_xreg_by_source['CH']['months_head_pad'],
    months_tail_pad=res_xreg_by_source['CH']['months_tail_pad'],
)
print(f"Full month sequence: {month_list_ch}")

In [ ]:
from regions.TF_Europe.scripts.dataset.topoclimatic_distance import _estimate_blur, _sinkhorn_distance

COMPARE_REGIONS = {
    'FR': {
        'color': NATURE_PALETTE['orange'],
        'label': 'FR (close)'
    },
    'ISL': {
        'color': NATURE_PALETTE['reddish_purple'],
        'label': 'ISL (far)'
    },
    'CENTRALASIA': {
        'color': NATURE_PALETTE['bluish_green'],
        'label': 'CENTRALASIA (furthest)'
    },
}

DISTANCE_TYPES = ['climate', 'topo', 'joint']

CLIMATE_COLS = MONTHLY_COLS
TOPO_COLS = STATIC_COLS

# CH training data and targets
df_CH = res_all_xreg_by_source['CH']['XREG_CH_TO_FR']['df_train']
df_targets = {
    'FR':
    res_all_xreg_by_source['CH']['XREG_CH_TO_FR']['df_test'],
    'ISL':
    res_all_xreg_by_source['CH']['XREG_CH_TO_ISL']['df_test'],
    'CENTRALASIA':
    res_all_xreg_by_source['CH']['XREG_CH_TO_CENTRALASIA']['df_test'],
}

# Recover padded month token sequence from the CH monthly data
all_months_in_data = df_CH['MONTHS'].unique().tolist()
print(f"All month tokens in CH data: {sorted(all_months_in_data)}")

# Define seasons using padded tokens
# Winter: accumulation season aug_→mar (padded pre + oct→mar)
# Summer: melt season apr→oct_ (apr→sep + padded post)
WINTER_TOKENS = ['aug_', 'sep_', 'oct', 'nov', 'dec', 'jan', 'feb', 'mar']
SUMMER_TOKENS = ['apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct_']

# Filter to tokens actually present in the data
SEASONS = {
    'Winter (Aug_-Mar)': [m for m in WINTER_TOKENS if m in all_months_in_data],
    'Summer (Apr-Oct_)': [m for m in SUMMER_TOKENS if m in all_months_in_data],
}

print(f"Winter tokens: {SEASONS['Winter (Aug_-Mar)']}")
print(f"Summer tokens: {SEASONS['Summer (Apr-Oct_)']}")

# Topo helper — ID-level aggregation
pure_static = [c for c in TOPO_COLS if c != 'ELEVATION_DIFFERENCE']


def _get_topo(df):
    parts = [df.groupby('ID')[pure_static].first()]
    if 'ELEVATION_DIFFERENCE' in TOPO_COLS:
        parts.append(df.groupby('ID')[['ELEVATION_DIFFERENCE']].mean())
    return pd.concat(parts, axis=1)[TOPO_COLS].to_numpy(dtype=np.float64)


RECOMPUTE_SEASONAL = True
seasonal_cache_path = BASE_DIR / "seasonal_distances_CH.pkl"

if not RECOMPUTE_SEASONAL and seasonal_cache_path.exists():
    with open(seasonal_cache_path, 'rb') as f:
        seasonal_distances = pickle.load(f)
    print("Loaded seasonal distances from cache")

else:
    seasonal_distances = {
        region: {
            season: {}
            for season in SEASONS
        }
        for region in COMPARE_REGIONS
    }

    for season_name, season_tokens in SEASONS.items():
        print(f"\n── {season_name} ──")
        print(f"   tokens: {season_tokens}")

        df_ch_season = df_CH[df_CH['MONTHS'].isin(season_tokens)]
        print(f"   CH rows: {len(df_ch_season)}")

        if len(df_ch_season) < 10:
            print("   Insufficient CH data, skipping season")
            continue

        Xm_ch = scaler_m.transform(
            df_ch_season[CLIMATE_COLS].to_numpy(dtype=np.float64))
        Xs_ch = scaler_s.transform(_get_topo(df_ch_season))
        id_codes_ch = pd.Categorical(df_ch_season['ID']).codes
        Xj_ch = np.hstack([Xm_ch, Xs_ch[id_codes_ch]]).astype(np.float32)

        for region in COMPARE_REGIONS:
            df_tgt = df_targets[region]
            df_tgt_season = df_tgt[df_tgt['MONTHS'].isin(season_tokens)]

            if len(df_tgt_season) < 10:
                print(
                    f"  {region}: insufficient data ({len(df_tgt_season)} rows), skipping"
                )
                for dt in DISTANCE_TYPES:
                    seasonal_distances[region][season_name][dt] = np.nan
                continue

            Xm_tgt = scaler_m.transform(
                df_tgt_season[CLIMATE_COLS].to_numpy(dtype=np.float64))
            Xs_tgt = scaler_s.transform(_get_topo(df_tgt_season))
            id_codes_tgt = pd.Categorical(df_tgt_season['ID']).codes
            Xj_tgt = np.hstack([Xm_tgt,
                                Xs_tgt[id_codes_tgt]]).astype(np.float32)

            blur_m_s = _estimate_blur(Xm_ch,
                                      Xm_tgt,
                                      blur_quantile_multiplier=0.1,
                                      seed=cfg.seed)
            blur_s_s = _estimate_blur(Xs_ch,
                                      Xs_tgt,
                                      blur_quantile_multiplier=0.1,
                                      seed=cfg.seed)
            blur_j_s = _estimate_blur(Xj_ch,
                                      Xj_tgt,
                                      blur_quantile_multiplier=0.1,
                                      seed=cfg.seed)

            d_climate = _sinkhorn_distance(Xm_ch,
                                           Xm_tgt,
                                           blur=blur_m_s,
                                           seed=cfg.seed)
            d_topo = _sinkhorn_distance(Xs_ch,
                                        Xs_tgt,
                                        blur=blur_s_s,
                                        seed=cfg.seed)
            d_joint = _sinkhorn_distance(Xj_ch,
                                         Xj_tgt,
                                         blur=blur_j_s,
                                         seed=cfg.seed)

            seasonal_distances[region][season_name]['climate'] = d_climate
            seasonal_distances[region][season_name]['topo'] = d_topo
            seasonal_distances[region][season_name]['joint'] = d_joint

            print(
                f"  {region:12s}: climate={d_climate:.4f}  topo={d_topo:.4f}  "
                f"joint={d_joint:.4f}  (target rows: {len(df_tgt_season)})")

    with open(seasonal_cache_path, 'wb') as f:
        pickle.dump(seasonal_distances, f)
    print(f"\nSaved to {seasonal_cache_path}")

In [ ]:
dist_labels = {'climate': 'Climate', 'topo': 'Topographic', 'joint': 'Joint'}

fig, axes = plt.subplots(1,
                         3,
                         figsize=nature_figsize(cols=3, height_mm=65),
                         sharey=False)

season_names = list(SEASONS.keys())
x = np.arange(len(season_names))
bar_w = 0.25
offsets = np.linspace(-(len(COMPARE_REGIONS) - 1) / 2,
                      (len(COMPARE_REGIONS) - 1) / 2,
                      len(COMPARE_REGIONS)) * bar_w

for ax, dist_type in zip(axes, DISTANCE_TYPES):
    for i, (region, cfg_r) in enumerate(COMPARE_REGIONS.items()):
        vals = [
            seasonal_distances[region][s].get(dist_type, np.nan)
            for s in season_names
        ]
        ax.bar(x + offsets[i],
               vals,
               width=bar_w * 0.9,
               color=cfg_r['color'],
               label=cfg_r['label'],
               alpha=0.85,
               zorder=3)

    ax.set_xticks(x)
    ax.set_xticklabels(season_names,
                       fontsize=NATURE_SPECS['font_max_pt'] - 1,
                       rotation=20,
                       ha='right')
    ax.set_ylabel(f'Sinkhorn distance ({dist_labels[dist_type]})',
                  fontsize=NATURE_SPECS['font_max_pt'])
    ax.set_title(dist_labels[dist_type],
                 fontsize=NATURE_SPECS['font_max_pt'] + 1,
                 fontweight='bold')
    if dist_type == 'climate':
        ax.legend(fontsize=NATURE_SPECS['font_max_pt'] - 1)
    apply_nature_style(ax)

fig.suptitle('Seasonal domain shift from CH — climate vs topo vs joint',
             fontsize=NATURE_SPECS['font_max_pt'] + 2,
             fontweight='bold')
fig.tight_layout()
fig.savefig('figures/paperTF/seasonal_domain_shift_decomposed.png',
            dpi=NATURE_SPECS['dpi'],
            bbox_inches='tight')
plt.show()

In [ ]:
CLIMATE_COLS_PLOT = [c for c in CLIMATE_COLS if c != 'POINT_BALANCE']
ALL_COLS_PLOT = CLIMATE_COLS_PLOT + TOPO_COLS

for season_name, season_tokens in SEASONS.items():
    df_ch_s  = df_CH[df_CH['MONTHS'].isin(season_tokens)].copy()
    df_fr_s  = df_targets['FR'][df_targets['FR']['MONTHS'].isin(season_tokens)].copy()
    df_isl_s = df_targets['ISL'][df_targets['ISL']['MONTHS'].isin(season_tokens)].copy()
    df_ca_s  = df_targets['CENTRALASIA'][
        df_targets['CENTRALASIA']['MONTHS'].isin(season_tokens)].copy()

    if min(len(df_ch_s), len(df_ca_s)) < 10:
        print(f"Skipping {season_name} — insufficient data")
        continue

    glaciers_to_plot = {
        'CH':          {'df': df_ch_s,  'color': NATURE_PALETTE['blue']},
        'ISL':         {'df': df_isl_s, 'color': NATURE_PALETTE['reddish_purple']},
        'CENTRALASIA': {'df': df_ca_s,  'color': NATURE_PALETTE['bluish_green']},
    }

    print(f"\n── {season_name} (tokens: {season_tokens}) ──")
    print(f"   CH={len(df_ch_s)}  FR={len(df_fr_s)}  "
          f"ISL={len(df_isl_s)}  CENTRALASIA={len(df_ca_s)}")

    plot_kde_pair(
        glaciers_to_plot=glaciers_to_plot,
        selected_cols=ALL_COLS_PLOT,  # climate + topo together
        save_prefix=f"seasonal_{season_name.split()[0].lower()}",
    )

## ML models
### Model datasets:

In [ ]:
logs_cache_dir = BASE_DIR / "LSTM_cache"
outputs_xreg_by_source = {}

# NaN check on augmented dataframes before building LSTM datasets
print("--- NaN check on augmented dataframes ---")
for src_region, res_xreg in res_xreg_by_source.items():
    for split_name, df in [("df_train_aug", res_xreg["df_train_aug"]),
                           ("df_test_aug", res_xreg["df_test_aug"])]:
        feat_cols = [c for c in MONTHLY_COLS + STATIC_COLS if c in df.columns]
        n_nan_feat = df[feat_cols].isna().sum().sum()
        n_nan_tgt = df["POINT_BALANCE"].isna().sum(
        ) if "POINT_BALANCE" in df.columns else "N/A"
        print(
            f"  {src_region} {split_name}: feature NaNs={n_nan_feat}, target NaNs={n_nan_tgt}"
        )

for src_region in SOURCE_REGIONS:
    src_members = SOURCE_MEMBERS[src_region]
    target_codes = [t for t in ALL_TARGETS if t not in src_members]

    outputs_xreg_by_source[src_region] = build_or_load_lstm_all_xreg(
        cfg=cfg,
        res_xreg=res_xreg_by_source[src_region],
        MONTHLY_COLS=MONTHLY_COLS,
        STATIC_COLS=STATIC_COLS,
        cache_dir=logs_cache_dir / src_region,
        ch_code=src_region,
        target_source_codes=target_codes,
        region_groups={},  # no group regions anymore
        force_recompute_train=True,
        force_recompute_tests=True,
    )
    print(f"{src_region} -> LSTM dataset keys:",
          list(outputs_xreg_by_source[src_region].keys()))

    # NaN/Inf check on the resulting datasets
    print(f"--- NaN/Inf check for {src_region} ---")
    for exp_key, assets in outputs_xreg_by_source[src_region].items():
        for ds_name in ["ds_train", "ds_test"]:
            ds = assets.get(ds_name)
            if ds is None:
                continue
            # Check directly on the underlying tensors (no iteration needed)
            checks = {"x_m": ds.Xm, "x_s": ds.Xs, "y": ds.y}
            any_issue = False
            for name, t in checks.items():
                n_nan = torch.isnan(t).sum().item()
                n_inf = torch.isinf(t).sum().item()
                if n_nan > 0 or n_inf > 0:
                    print(
                        f"  {exp_key} {ds_name} {name}: NaNs={n_nan}, Infs={n_inf}"
                    )
                    any_issue = True
            if not any_issue:
                print(f"  {exp_key} {ds_name}: OK")

### Train model:

In [ ]:
def train_or_load_one_source_model(
        cfg,
        key: str,
        lstm_assets: dict,
        best_params: dict,
        device,
        models_dir="models",
        prefix="lstm_xreg",
        train_flag=True,
        force_retrain=False,
        epochs=150,
        batch_size_train=64,
        batch_size_val=128,
        verbose=True,
        date=None,
        model_type="lstm",  # <-- new: "lstm" or "transformer"
):
    """Train or load a single source-region model. No test set needed."""
    assert model_type in ("lstm", "transformer"), \
        f"model_type must be 'lstm' or 'transformer', got '{model_type}'"

    # ---- resolve model class once ----
    model_cls = mbm.models.LSTM_MB if model_type == "lstm" else mbm.models.Transformer_MB

    run_date = datetime.now().strftime("%Y-%m-%d") if date is None else date
    os.makedirs(models_dir, exist_ok=True)
    model_filename = os.path.join(models_dir, f"{prefix}_{key}_{run_date}.pt")

    # ---- these are the only two lines that used to be hardcoded ----
    model = model_cls.build_model_from_params(cfg,
                                              best_params,
                                              device,
                                              verbose=verbose)
    loss_fn = model_cls.resolve_loss_fn(best_params)

    # everything below is unchanged
    if train_flag and (not force_retrain) and os.path.exists(model_filename):
        state = torch.load(model_filename, map_location=device)
        model.load_state_dict(state)
        return model, model_filename, None

    if not train_flag and not os.path.exists(model_filename):
        raise FileNotFoundError(
            f"No checkpoint found for {key}: {model_filename}")

    if not train_flag and os.path.exists(model_filename):
        state = torch.load(model_filename, map_location=device)
        model.load_state_dict(state)
        return model, model_filename, None

    # --- Train ---
    mbm.utils.seed_all(cfg.seed)

    ds_train = lstm_assets["ds_train"]
    train_idx = lstm_assets["train_idx"]
    val_idx = lstm_assets["val_idx"]

    ds_train_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ds_train)

    train_dl, val_dl = ds_train_copy.make_loaders(
        train_idx=train_idx,
        val_idx=val_idx,
        batch_size_train=batch_size_train,
        batch_size_val=batch_size_val,
        seed=cfg.seed,
        fit_and_transform=True,
        shuffle_train=True,
        use_weighted_sampler=True,
        verbose=verbose,
    )

    if os.path.exists(model_filename):
        os.remove(model_filename)
        if verbose:
            print(f"Deleted existing model file: {model_filename}")

    history, best_val, best_state = model.train_loop(
        device=device,
        train_dl=train_dl,
        val_dl=val_dl,
        epochs=epochs,
        lr=best_params["lr"],
        weight_decay=best_params["weight_decay"],
        clip_val=1,
        sched_factor=0.5,
        sched_patience=6,
        sched_threshold=0.01,
        sched_threshold_mode="rel",
        sched_cooldown=1,
        sched_min_lr=1e-6,
        es_patience=15,
        es_min_delta=1e-4,
        log_every=5,
        verbose=verbose,
        save_best_path=model_filename,
        loss_fn=loss_fn,
    )

    if verbose:
        plot_history_lstm(history)

    state = torch.load(model_filename, map_location=device)
    model.load_state_dict(state)

    return model, model_filename, {"history": history, "best_val": best_val}


def prepare_test_loader_for_target(
    cfg,
    model,
    lstm_assets_src: dict,  # needs ds_train (for scaler fitting)
    lstm_assets_tgt: dict,  # needs ds_test
    batch_size_test=128,
):
    """Given a trained model and a target's test assets, build the test loader."""
    ds_train_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        lstm_assets_src["ds_train"])
    ds_test_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        lstm_assets_tgt["ds_test"])

    # fit scaler on train, apply to test
    ds_train_copy.make_loaders(
        train_idx=lstm_assets_src["train_idx"],
        val_idx=lstm_assets_src["val_idx"],
        fit_and_transform=True,
        seed=cfg.seed,
    )

    test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
        ds_test_copy, ds_train_copy, batch_size=batch_size_test, seed=cfg.seed)

    return test_dl, ds_test_copy

#### Transformer:

In [ ]:
gs_logs_dir = Path(cfg.dataPath) / path_cache / "TF_REGION/logs/Transformer_GS"
logs_gs_transformer = pd.read_csv(
    os.path.join(gs_logs_dir, "transformer_gs_pooled_2026-05-20.csv"))
logs_gs_transformer.sort_values("valid_loss")

In [ ]:
# best by validation loss
best_row = logs_gs_transformer.sort_values("valid_loss").iloc[1]
print("Best config:")
print(best_row[[
    "d_model", "nhead", "num_layers", "dim_feedforward", "dropout",
    "static_layers", "static_hidden", "static_dropout", "head_dropout", "lr",
    "weight_decay", "valid_loss", "val_rmse_a", "val_rmse_w"
]])

best_params_gs = {
    "Fm":
    int(best_row["Fm"]),
    "Fs":
    int(best_row["Fs"]),
    "d_model":
    int(best_row["d_model"]),
    "nhead":
    int(best_row["nhead"]),
    "num_layers":
    int(best_row["num_layers"]),
    "dim_feedforward":
    int(best_row["dim_feedforward"]),
    "dropout":
    float(best_row["dropout"]),
    "static_layers":
    int(best_row["static_layers"]),
    "static_hidden": (None if pd.isna(best_row["static_hidden"]) else int(
        best_row["static_hidden"])),
    "static_dropout": (None if pd.isna(best_row["static_dropout"]) else float(
        best_row["static_dropout"])),
    "head_dropout":
    float(best_row["head_dropout"]),
    "lr":
    float(best_row["lr"]),
    "weight_decay":
    float(best_row["weight_decay"]),
    "two_heads":
    False,
    "loss_spec":
    None,
    "T_max":
    32,
}

print("\nbest_params_gs:")
for k, v in best_params_gs.items():
    print(f"  {k}: {v}")

In [ ]:
transformer_params = dict(best_params_gs)
transformer_params['Fm'] = len(MONTHLY_COLS)
transformer_params['Fs'] = len(STATIC_COLS)

models_by_source_transformer = {}
infos_by_source_transformer = {}

TRAIN_FLAG = False
MODEL_DATE = "2026-05-29"

for src_region in SOURCE_REGIONS:
    any_key = next(iter(outputs_xreg_by_source[src_region]))
    train_assets = outputs_xreg_by_source[src_region][any_key]

    model, model_path, info = train_or_load_one_source_model(
        cfg=cfg,
        key=src_region,
        lstm_assets=train_assets,
        best_params=transformer_params,
        device=device,
        models_dir=BASE_DIR / "models" / src_region,
        prefix=f"transformer_xreg_{src_region}",  # <-- changed
        train_flag=TRAIN_FLAG,
        force_retrain=True,
        epochs=150,
        date=MODEL_DATE,
        model_type="transformer",  # <-- new
    )

    models_by_source_transformer[src_region] = model
    infos_by_source_transformer[src_region] = {
        "model_path": model_path,
        **(info or {})
    }
    print(f"{src_region} -> model trained/loaded from {model_path}")

In [ ]:
df_metrics_by_source_transformer = {}
preds_by_source_transformer = {}
figs_by_source_transformer = {}

DISPLAY_ORDER = ["FR", "ISL", "CENTRALASIA"]

# Preferred display order for target regions — any not listed appear at the end
for src_region in SOURCE_REGIONS:
    print(f"\nEvaluating source region: {src_region}")
    target_keys = list(outputs_xreg_by_source[src_region].keys())
    models_by_key = {
        key: models_by_source_transformer[src_region]
        for key in target_keys
    }

    # Derive custom_order from actual keys, sorted by preferred display order
    available_targets = [k.split("_TO_")[-1] for k in target_keys]
    custom_order = [t for t in DISPLAY_ORDER if t in available_targets]
    # append any targets not in DISPLAY_ORDER at the end
    custom_order += [t for t in available_targets if t not in custom_order]

    df_metrics, preds_by_key, figs_by_key, fig_grid = evaluate_all_models(
        cfg=cfg,
        models_by_key=models_by_key,
        lstm_assets_by_key=outputs_xreg_by_source[src_region],
        device=device,
        save_dir=f"figures/eval_xreg/{src_region}",
        ax_xlim=(-16, 10),
        ax_ylim=(-16, 10),
        grid_shape=(1, 3),
        grid_figsize=(25, 12),
        domain_shifts=all_shifts_by_source[src_region],
        complement_key=f'XREG_{src_region}_TO_',
        custom_order=custom_order,
    )

    df_metrics_by_source_transformer[src_region] = df_metrics
    preds_by_source_transformer[src_region] = preds_by_key
    figs_by_source_transformer[src_region] = figs_by_key

In [ ]:
color_palette = {
    'CH': NATURE_PALETTE["blue"],
    'NOR': NATURE_PALETTE["vermillion"],
    'ALA': NATURE_PALETTE["bluish_green"],
    'ISL': NATURE_PALETTE["reddish_purple"]
}
panel_titles = [
    "(a)  All variables ", "(b) ERA5 Land & Elev. diff.",
    "(c)  Topographic variables"
]

EXCLUDE_TARGETS = []

# Combine across all sources
df_metrics_all = pd.concat(df_metrics_by_source_transformer.values())
all_shifts_all = {}
for src_region in SOURCE_REGIONS:
    all_shifts_all.update(all_shifts_by_source[src_region])

# Filter out excluded targets
if EXCLUDE_TARGETS:
    mask = ~df_metrics_all.index.str.contains("|".join(EXCLUDE_TARGETS))
    df_metrics_all = df_metrics_all[mask]
    all_shifts_all = {
        k: v
        for k, v in all_shifts_all.items()
        if not any(tgt in k for tgt in EXCLUDE_TARGETS)
    }

fig, _ = plot_region_shift_vs_performance_single_d(
    df_metrics_all,
    all_shifts_all,
    distance_variant="sinkhorn",
    performance_cols=["RMSE_annual"],
    joint_variant="true",
    blur_m=blur_m,
    blur_s=blur_s,
    color_palette=color_palette,
    panel_titles=panel_titles,
    suptitle="Transformer: Region-level domain shift vs transfer performance")

# save figure
plt.savefig(
    "figures/paperTF/section1_shift_performance.png",
    dpi=NATURE_SPECS["dpi"],
    bbox_inches="tight",
)

## Separate into CA regions:

In [ ]:
import geopandas as gpd

# Load RGI shapefile for CA (region 13) to get O2Region
rgi_gdf = gpd.read_file(
    cfg.dataPath / RGI_V6_ROOT /
    RGI_REGIONS['13']['folder'] /
    RGI_REGIONS['13']['file']
)[['RGIId', 'O2Region']]

# Build glacier → O2Region mapping
glacier_dict_ca = (
    dfs['13'][['GLACIER', 'RGIId']].drop_duplicates()
    .merge(rgi_gdf, on='RGIId', how='left')
    .set_index('GLACIER')
    .to_dict('index')
)
o2_map = {k: v['O2Region'] for k, v in glacier_dict_ca.items()}

# Add O2Region to CENTRALASIA monthly data
df_ca = res_all_xreg_by_source['CH']['XREG_CH_TO_CENTRALASIA']['df_test'].copy()
df_ca['O2Region'] = df_ca['GLACIER'].map(o2_map)

print("O2Region distribution in CA test data:")
print(df_ca.groupby('O2Region').agg(
    n_rows=('GLACIER', 'size'),
    n_glaciers=('GLACIER', 'nunique')
))

# Subregion dataframes
df_ca_12 = df_ca[df_ca['O2Region'].isin(['1', '2'])].copy()
df_ca_3  = df_ca[df_ca['O2Region'] == '3'].copy()
df_ca_4  = df_ca[df_ca['O2Region'] == '4'].copy()

print(f"\nCA_12 : {len(df_ca_12):,} rows, {df_ca_12['GLACIER'].nunique()} glaciers")
print(f"CA_3  : {len(df_ca_3):,} rows,  {df_ca_3['GLACIER'].nunique()} glaciers")
print(f"CA_4  : {len(df_ca_4):,} rows,  {df_ca_4['GLACIER'].nunique()} glaciers")
print(f"Unmapped: {df_ca['O2Region'].isna().sum()} rows")

In [ ]:
from regions.TF_Europe.scripts.dataset.topoclimatic_distance import _estimate_blur, _sinkhorn_distance

CA_SUBREGIONS = {
    'CA_12': {'df': df_ca_12, 'color': NATURE_PALETTE['blue'],         'label': 'CA 1+2 (Western/Eastern Tien Shan)'},
    'CA_3':  {'df': df_ca_3,  'color': NATURE_PALETTE['vermillion'],   'label': 'CA 3 (Pamir/Karakoram)'},
    'CA_4':  {'df': df_ca_4,  'color': NATURE_PALETTE['bluish_green'], 'label': 'CA 4 (Tibetan Plateau fringe)'},
}

DISTANCE_TYPES = ['climate', 'topo', 'joint']

pure_static_ca = [c for c in TOPO_COLS if c != 'ELEVATION_DIFFERENCE']

def _get_topo_ca(df):
    parts = [df.groupby('ID')[pure_static_ca].first()]
    if 'ELEVATION_DIFFERENCE' in TOPO_COLS:
        parts.append(df.groupby('ID')[['ELEVATION_DIFFERENCE']].mean())
    return pd.concat(parts, axis=1)[TOPO_COLS].to_numpy(dtype=np.float64)

RECOMPUTE_CA_SEASONAL = True
ca_seasonal_cache = BASE_DIR / "seasonal_distances_CA_subregions.pkl"

if not RECOMPUTE_CA_SEASONAL and ca_seasonal_cache.exists():
    with open(ca_seasonal_cache, 'rb') as f:
        ca_seasonal_distances = pickle.load(f)
    print("Loaded CA subregion seasonal distances from cache")
else:
    ca_seasonal_distances = {
        sub: {season: {} for season in SEASONS}
        for sub in CA_SUBREGIONS
    }

    for season_name, season_tokens in SEASONS.items():
        print(f"\n── {season_name} ──")

        df_ch_season = df_CH[df_CH['MONTHS'].isin(season_tokens)]
        if len(df_ch_season) < 10:
            print("  Insufficient CH data, skipping")
            continue

        Xm_ch = scaler_m.transform(
            df_ch_season[list(CLIMATE_COLS)].to_numpy(dtype=np.float64))
        Xs_ch = scaler_s.transform(_get_topo_ca(df_ch_season))
        id_codes_ch = pd.Categorical(df_ch_season['ID']).codes
        Xj_ch = np.hstack([Xm_ch, Xs_ch[id_codes_ch]]).astype(np.float32)

        for sub_name, sub_cfg in CA_SUBREGIONS.items():
            df_sub_season = sub_cfg['df'][
                sub_cfg['df']['MONTHS'].isin(season_tokens)]

            if len(df_sub_season) < 10:
                print(f"  {sub_name}: insufficient data ({len(df_sub_season)} rows), skipping")
                for dt in DISTANCE_TYPES:
                    ca_seasonal_distances[sub_name][season_name][dt] = np.nan
                continue

            Xm_tgt = scaler_m.transform(
                df_sub_season[list(CLIMATE_COLS)].to_numpy(dtype=np.float64))
            Xs_tgt = scaler_s.transform(_get_topo_ca(df_sub_season))
            id_codes_tgt = pd.Categorical(df_sub_season['ID']).codes
            Xj_tgt = np.hstack([Xm_tgt, Xs_tgt[id_codes_tgt]]).astype(np.float32)

            blur_m_s = _estimate_blur(Xm_ch, Xm_tgt, blur_quantile_multiplier=0.1, seed=cfg.seed)
            blur_s_s = _estimate_blur(Xs_ch, Xs_tgt, blur_quantile_multiplier=0.1, seed=cfg.seed)
            blur_j_s = _estimate_blur(Xj_ch, Xj_tgt, blur_quantile_multiplier=0.1, seed=cfg.seed)

            d_climate = _sinkhorn_distance(Xm_ch,  Xm_tgt,  blur=blur_m_s, seed=cfg.seed)
            d_topo    = _sinkhorn_distance(Xs_ch,  Xs_tgt,  blur=blur_s_s, seed=cfg.seed)
            d_joint   = _sinkhorn_distance(Xj_ch,  Xj_tgt,  blur=blur_j_s, seed=cfg.seed)

            ca_seasonal_distances[sub_name][season_name]['climate'] = d_climate
            ca_seasonal_distances[sub_name][season_name]['topo']    = d_topo
            ca_seasonal_distances[sub_name][season_name]['joint']   = d_joint

            print(f"  {sub_name}: climate={d_climate:.4f}  topo={d_topo:.4f}  "
                  f"joint={d_joint:.4f}  (target rows: {len(df_sub_season)})")

    with open(ca_seasonal_cache, 'wb') as f:
        pickle.dump(ca_seasonal_distances, f)
    print(f"\nSaved to {ca_seasonal_cache}")

In [ ]:
# ── Bar chart: seasonal distances per CA subregion ────────────────────────────
dist_labels = {'climate': 'Climate', 'topo': 'Topographic', 'joint': 'Joint'}
season_names = list(SEASONS.keys())
x = np.arange(len(season_names))
bar_w = 0.25
offsets = np.linspace(-(len(CA_SUBREGIONS)-1)/2,
                       (len(CA_SUBREGIONS)-1)/2,
                        len(CA_SUBREGIONS)) * bar_w

fig, axes = plt.subplots(1, 3, figsize=nature_figsize(cols=3, height_mm=65), sharey=False)

for ax, dist_type in zip(axes, DISTANCE_TYPES):
    for i, (sub_name, sub_cfg) in enumerate(CA_SUBREGIONS.items()):
        vals = [
            ca_seasonal_distances[sub_name][s].get(dist_type, np.nan)
            for s in season_names
        ]
        ax.bar(x + offsets[i], vals, width=bar_w * 0.9,
               color=sub_cfg['color'], label=sub_cfg['label'],
               alpha=0.85, zorder=3)

    ax.set_xticks(x)
    ax.set_xticklabels(season_names, fontsize=NATURE_SPECS['font_max_pt'] - 1,
                       rotation=20, ha='right')
    ax.set_ylabel(f'Sinkhorn distance to CH ({dist_labels[dist_type]})',
                  fontsize=NATURE_SPECS['font_max_pt'])
    ax.set_title(dist_labels[dist_type],
                 fontsize=NATURE_SPECS['font_max_pt'] + 1, fontweight='bold')
    if dist_type == 'climate':
        ax.legend(fontsize=NATURE_SPECS['font_max_pt'] - 2)
    apply_nature_style(ax)

fig.suptitle('Seasonal domain shift from CH — Central Asia subregions',
             fontsize=NATURE_SPECS['font_max_pt'] + 2, fontweight='bold')
fig.tight_layout()
fig.savefig('figures/paperTF/seasonal_domain_shift_CA_subregions.png',
            dpi=NATURE_SPECS['dpi'], bbox_inches='tight')
plt.show()

# ── KDE per season, CA subregions ────────────────────────────────────────────
CLIMATE_COLS_PLOT = list(CLIMATE_COLS)

for season_name, season_tokens in SEASONS.items():
    df_ch_s = df_CH[df_CH['MONTHS'].isin(season_tokens)].copy()

    sub_dfs = {
        sub_name: sub_cfg['df'][
            sub_cfg['df']['MONTHS'].isin(season_tokens)].copy()
        for sub_name, sub_cfg in CA_SUBREGIONS.items()
    }

    if any(len(df) < 10 for df in [df_ch_s] + list(sub_dfs.values())):
        print(f"Skipping {season_name} — insufficient data in one or more subregions")
        continue

    glaciers_to_plot = {'CH': {'df': df_ch_s, 'color': NATURE_PALETTE['blue']}}
    glaciers_to_plot.update({
        sub_name: {'df': sub_df, 'color': CA_SUBREGIONS[sub_name]['color']}
        for sub_name, sub_df in sub_dfs.items()
    })

    print(f"\n── {season_name} ──")
    for k, v in glaciers_to_plot.items():
        print(f"   {k}: {len(v['df'])} rows")

    plot_kde_pair(
        glaciers_to_plot=glaciers_to_plot,
        selected_cols=CLIMATE_COLS_PLOT + TOPO_COLS,
        save_prefix=f"ca_subregions_{season_name.split()[0].lower()}",
    )

In [ ]:
CA_SUBREGION_DATASETS = {}

for sub_name, sub_cfg in CA_SUBREGIONS.items():
    df_sub     = sub_cfg['df']
    df_sub_aug = res_all_xreg_by_source['CH']['XREG_CH_TO_CENTRALASIA']['df_test_aug']
    df_sub_aug = df_sub_aug[df_sub_aug['GLACIER'].isin(df_sub['GLACIER'].unique())]

    ds_sub = build_combined_LSTM_dataset(
        df_loss=df_sub,
        df_full=df_sub_aug,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        months_head_pad=res_xreg_by_source['CH']['months_head_pad'],
        months_tail_pad=res_xreg_by_source['CH']['months_tail_pad'],
        normalize_target=True,
        expect_target=True,
        show_progress=False,
    )

    CA_SUBREGION_DATASETS[sub_name] = ds_sub
    print(f"{sub_name}: {len(ds_sub)} sequences "
          f"({ds_sub.iw.sum().item()} winter | {ds_sub.ia.sum().item()} annual)")

In [ ]:
# Build per-subregion assets in the format evaluate_all_models expects
ca_subregion_assets = {}
ca_subregion_models = {}

for sub_name, ds_sub in CA_SUBREGION_DATASETS.items():
    key = f"XREG_CH_TO_{sub_name}"
    ca_subregion_assets[key] = {
        **outputs_xreg_by_source['CH']['XREG_CH_TO_CENTRALASIA'],
        'ds_test': ds_sub,
    }
    ca_subregion_models[key] = models_by_source_transformer['CH']

df_metrics_ca_sub, preds_ca_sub, figs_ca_sub, fig_grid_ca_sub = evaluate_all_models(
    cfg=cfg,
    models_by_key=ca_subregion_models,
    lstm_assets_by_key=ca_subregion_assets,
    device=device,
    save_dir="figures/eval_xreg/CH_CA_subregions",
    ax_xlim=(-16, 10),
    ax_ylim=(-16, 10),
    grid_shape=(1, 3),
    grid_figsize=(25, 12),
    domain_shifts=all_shifts_by_source['CH'],
    complement_key='XREG_CH_TO_',
    custom_order=['CA_12', 'CA_3', 'CA_4'],
)

print(df_metrics_ca_sub)

In [ ]:
# ── Compute domain shifts for CA subregions ───────────────────────────────────
ca_sub_shifts = {}

for sub_name, sub_cfg in CA_SUBREGIONS.items():
    key = f"XREG_CH_TO_{sub_name}"
    shift = compute_domain_shift(
        df_src=df_CH,
        df_tgt=sub_cfg['df'],
        monthly_cols=list(CLIMATE_COLS),
        static_cols=list(TOPO_COLS),
        scaler_m=scaler_m,
        scaler_s=scaler_s,
        blur_m=blur_m,
        blur_s=blur_s,
        blur_joint=blur_joint,
        device=str(device),
        seed=cfg.seed,
    )
    ca_sub_shifts[key] = shift
    print(f"{key}: D_sinkhorn_joint={shift['D_sinkhorn_joint']:.4f}  "
          f"D_sinkhorn_climate={shift['D_sinkhorn_climate']:.4f}  "
          f"D_sinkhorn_topo={shift['D_sinkhorn_topo']:.4f}")

# ── Merge into the existing shifts and metrics dicts ─────────────────────────
color_palette = {
    'CH':   NATURE_PALETTE["blue"],
    'NOR':  NATURE_PALETTE["vermillion"],
    'ALA':  NATURE_PALETTE["bluish_green"],
    'ISL':  NATURE_PALETTE["reddish_purple"],
    'CA_12': NATURE_PALETTE["orange"],
    'CA_3':  NATURE_PALETTE["sky_blue"],
    'CA_4':  NATURE_PALETTE["yellow"],
}
panel_titles = [
    "(a)  All variables",
    "(b) ERA5 Land & Elev. diff.",
    "(c)  Topographic variables",
]

EXCLUDE_TARGETS = []

df_metrics_all = pd.concat(df_metrics_by_source_transformer.values())
all_shifts_all = {}
for src_region in SOURCE_REGIONS:
    all_shifts_all.update(all_shifts_by_source[src_region])

# Add CA subregion shifts and metrics
all_shifts_all.update(ca_sub_shifts)
df_metrics_all = pd.concat([df_metrics_all, df_metrics_ca_sub])

if EXCLUDE_TARGETS:
    mask = ~df_metrics_all.index.str.contains("|".join(EXCLUDE_TARGETS))
    df_metrics_all = df_metrics_all[mask]
    all_shifts_all = {
        k: v for k, v in all_shifts_all.items()
        if not any(tgt in k for tgt in EXCLUDE_TARGETS)
    }

fig, _ = plot_region_shift_vs_performance_single_d(
    df_metrics_all,
    all_shifts_all,
    distance_variant="sinkhorn",
    performance_cols=["RMSE_annual"],
    joint_variant="true",
    blur_m=blur_m,
    blur_s=blur_s,
    color_palette=color_palette,
    panel_titles=panel_titles,
    suptitle="Transformer: Region-level domain shift vs transfer performance\n(incl. CA subregions)",
)

plt.savefig(
    "figures/paperTF/section1_shift_performance_ca_sub.png",
    dpi=NATURE_SPECS["dpi"],
    bbox_inches="tight",
)